# Embeddings

임베딩은 텍스트의 의미를 실수 벡터로 바꾸는 표현 방식이다. 의미가 가까운 문장은 벡터 공간에서도 가까워지므로 검색과 추천에 사용할 수 있다.

이 노트북에서는 문장 벡터를 만든다. 리뷰 벡터를 저장한 뒤 질의 벡터와의 cosine similarity로 가까운 리뷰를 찾는다.


## 모델과 벡터 차원

이 노트북은 비용과 저장 공간을 고려해 `text-embedding-3-small`을 사용한다.

- `text-embedding-3-small`의 기본 출력은 1,536차원이다.
- `text-embedding-3-large`의 기본 출력은 3,072차원이다.
- 두 모델 모두 `dimensions`로 출력 차원을 줄일 수 있다.
- 차원을 줄이면 저장·검색 비용과 검색 품질이 함께 달라질 수 있다.
- 최종 모델과 차원은 실제 서비스 검증 데이터로 결정한다.

공식 문서는 다음과 같다.

- [Embeddings 가이드](https://developers.openai.com/api/docs/guides/embeddings)
- [Embeddings 생성 API](https://developers.openai.com/api/reference/resources/embeddings/methods/create)
- [text-embedding-3-small 모델 문서](https://developers.openai.com/api/docs/models/text-embedding-3-small)


## MTEB 벤치마크 참고

MTEB는 분류, 군집화, 검색, 의미 유사도 등 여러 임베딩 작업을 평가하는 벤치마크 모음이다. 리더보드는 데이터셋과 평가 구성에 따라 달라지므로 특정 모델의 고정 순위를 보장하지 않는다.

- 검색 과제는 nDCG@k, 재정렬 과제는 MRR@k 또는 MAP 같은 지표를 사용한다.
- 서비스에서 사용할 언어, 문서 길이, 질의 유형과 유사한 검증 데이터로 모델을 비교한다.
- MTEB 리더보드: https://huggingface.co/spaces/mteb/leaderboard


### API 클라이언트 준비

이미 설정한 `.env`를 불러온다. 키 값은 출력하지 않는다.


In [1]:
import os

from dotenv import find_dotenv, load_dotenv
from parso import normalizer

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("08_llm 프로젝트 최상위의 .env 파일을 확인한다.")
load_dotenv(dotenv_path, override=False)

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env의 OPENAI_API_KEY를 확인한다.")


### 단일 문장을 1,536차원 벡터로 변환하기

문자열 하나를 길이 1의 목록으로 `input`에 전달하면 응답의 `data[0].embedding`에 실수 벡터가 들어간다. 이 벡터는 다음 셀의 배치 변환 함수와 같은 형식이다.

모델 문서: https://developers.openai.com/api/docs/models/text-embedding-3-small


In [2]:
from openai import OpenAI

# OpenAI()는 환경 변수에서 인증 정보를 읽는다.
client = OpenAI()
text = '임베딩 예시 문장'

response = client.embeddings.create(
    model='text-embedding-3-small',
    input=[text],
)
# data[0]은 입력 목록의 첫 문장에 해당한다.
embedding = response.data[0].embedding
print(len(embedding))


1536


### 여러 문장을 한 번에 임베딩하기

문자열 목록을 줄바꿈만 공백으로 바꿔 의미를 보존한 채 API에 전달한다. 응답 벡터의 순서는 입력 문장 순서와 같으며, 결과는 리뷰 데이터의 `embedding` 열에 저장된다.

특수문자나 개행을 일괄 삭제하면 품질이 높아진다고 단정할 수 없다. 서비스 데이터의 의미를 유지하는 정규화만 적용하고 검색 결과로 효과를 확인한다.


In [4]:
import numpy as np

def texts_to_embedding(texts, model='text-embedding-3-small'):
    # 줄바꿈만 공백으로 변환하여 문장경계를 유지한 입력목록 만들기
    normalized_texts = [text.replace('\n', ' ') for text in texts]

    # 각 입력 문장과 같은 순서의 embedding목록을 응답으로 받기
    response = client.embeddings.create(
        model=model,
        input=normalized_texts,
    )

    # response.data: 문장 묶음 -> 벡터 묶음
    return [item.embedding for item in response.data]


# 2문장을 입력하여 결과 반환받기
sample_texts = [
    'hello world',
    'goodbye world'
]

output = texts_to_embedding(sample_texts)
print(np.array(output).shape) # (2, 1536) 2문장, 각 1536벡터

(2, 1536)


## 음식 리뷰 유사도 검색

검색은 문서와 질의를 같은 모델로 임베딩한 뒤 cosine similarity가 높은 문서를 반환하는 과정이다. 아래에서는 리뷰의 제목과 본문을 결합해 문서 하나당 벡터 하나를 만든다.


### 리뷰 CSV 내려받기

이 셀은 외부 파일 ID에서 `fine_food_reviews_1k.csv`를 받는다. 네트워크와 외부 파일 상태에 따라 다운로드가 실패할 수 있다.

다운로드가 끝나면 다음 셀의 `read_csv`가 같은 이름의 CSV를 DataFrame으로 읽는다.


In [3]:
import subprocess

subprocess.run(
    [
        "gdown",
        "--output",
        "fine_food_reviews_1k.csv",
        "1tSQZQFYD64_mrL9CjDcn6KruZp7_smuD",
    ],
    check=True,
)


FileNotFoundError: [WinError 2] 지정된 파일을 찾을 수 없습니다

### 리뷰 행을 DataFrame으로 읽기

CSV의 각 행은 리뷰 하나이며 `Summary`와 `Text` 열이 다음 결합 단계의 입력이다. `head()`는 열 이름과 일부 행을 보여 주어 문자열 열이 올바르게 읽혔는지 확인한다.


In [ ]:
import pandas as pd

# 첫 번째 CSV 열은 행 인덱스로 사용한다.
df = pd.read_csv('fine_food_reviews_1k.csv', index_col=0)
df.head()


### 리뷰 데이터의 열과 결측치 확인

`info()`는 행 수, 열 이름, 결측이 아닌 값의 수를 보여 준다. `Summary` 또는 `Text`에 결측치가 있으면 문자열 결합 전에 처리 규칙을 정해야 한다.


In [ ]:
df.info()


### 제목과 본문을 하나의 검색 문서로 결합하기

`str.strip()`은 앞뒤 공백만 정리하고 제목과 본문 사이에는 `; `를 넣는다. 생성한 `combined` 문자열은 다음 API 요청의 입력이며, 나중에는 검색 결과의 인덱스가 된다.


In [ ]:
# 제목과 본문을 구분자로 이어 리뷰당 검색 문서 하나를 만든다.
df['combined'] = df['Summary'].str.strip() + '; ' + df['Text'].str.strip()
df[['combined']].head()


### 리뷰 문서를 벡터 열로 저장하기

`combined` 열의 문자열 목록을 배치로 보내면 각 행에 대응하는 임베딩 목록이 돌아온다. 이 API 호출은 네트워크와 인증이 필요하며, 반환한 벡터 열은 다음 검색 인덱스의 입력이다.


### 저장된 벡터의 값과 길이 살펴보기

이 셀은 첫 리뷰 벡터를 확인하는 용도이다. 전체 벡터 목록을 출력하면 너무 길어지므로 첫 행의 길이와 앞부분만 확인한다.


In [ ]:
first_embedding = df['embedding'].iloc[0]
print(len(first_embedding), first_embedding[:5])


### 검색용 벡터 인덱스 만들기

`embed_df`는 벡터를 열로, 결합 리뷰 문장을 인덱스로 둔다. 다음 검색 함수는 이 DataFrame에 문서별 cosine similarity 열을 추가하고 상위 행을 반환한다.


In [ ]:
# 리뷰 원문을 인덱스로 두어 검색 결과에서 바로 읽는다.
embed_df = df[['embedding']].copy()
embed_df.index = df['combined']
embed_df.head()


### cosine similarity로 상위 리뷰 찾기

cosine similarity는 두 벡터 방향의 유사도를 계산하며 1에 가까울수록 방향이 비슷하다. 질의와 문서는 반드시 같은 임베딩 모델로 변환해야 차원과 벡터 공간이 일치한다.

함수는 질의 문자열을 1개 벡터로 바꾸고, 모든 문서 벡터와의 점수를 계산한 뒤 높은 순서로 `top_n`개를 반환한다.


### 다른 질의로 검색 결과 비교하기

같은 저장 벡터에 다른 질의를 넣으면 문서 임베딩을 다시 만들지 않고 순위만 다시 계산한다. 오탈자나 짧은 질의에서 결과가 약하면 질의 정규화, 데이터 품질, 모델 선택을 검증 데이터로 점검한다.


In [ ]:
review_search('bad delivery', embed_df)
